# IDS — Exploratory Data Analysis

Load CICIDS-2018, inspect class distribution, feature correlations, and threshold sensitivity.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd, numpy as np
from configs.loader import get_config
from training.dataset import load_parquet
cfg = get_config('../configs/config.yaml')
df = load_parquet(cfg.training['dataset_gcs_path'])
print(df.shape); print(df['label'].value_counts())

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
df['label'].value_counts().plot(kind='bar'); plt.title('Class distribution'); plt.tight_layout(); plt.show()

In [ ]:
corr = df[cfg.numerical_cols].corr()
plt.figure(figsize=(14,12)); sns.heatmap(corr, annot=False, cmap='coolwarm', center=0)
plt.title('Feature correlations'); plt.tight_layout(); plt.show()

In [ ]:
# FNR/FPR threshold sweep (run after training)
from inference.serving.model_registry import ModelRegistry
from training.dataset import prepare_splits
from features.preprocessor import FeaturePreprocessor
from training.metrics import compute_metrics
reg = ModelRegistry.get()
pre = reg.preprocessor
splits = prepare_splits(df, pre, apply_smote=False)
X_test, y_test = splits['test']
scores = reg.xgb.predict_proba(X_test)
rows = [{'threshold':t, **compute_metrics(y_test, scores, t)} for t in [0.2,0.3,0.35,0.4,0.5,0.6,0.75,0.9]]
print(pd.DataFrame(rows)[['threshold','fnr','fpr','f1']].to_string(index=False))